# Milestone 3 — Manual Labelling trên Google Colab

**Nhánh:** `m3-manual-labeling`  
**MSSV:** `24127465`

Notebook này thực hiện quy trình **gán nhãn thủ công 100% bằng spreadsheet** cho Milestone 3.

Flow chính:

```text
M2 BibTeX + M1 references + selection.csv
            ↓
      label_sheet.py make
            ↓
mỗi publication → <paper>.csv + <paper>_references.csv
            ↓
Google Sheets: người dùng tự đối chiếu
            ↓
ref_arxiv_id = ground truth
best_suggestion = chỉ là gợi ý
            ↓
      label_sheet.py collect
            ↓
labels_manual_24127465.csv
```

## Nguyên tắc quan trọng

- `best_suggestion` được ghi vào CSV để hỗ trợ tìm kiếm.
- `best_suggestion` **không phải ground truth** và không tự động điền `ref_arxiv_id`.
- `ref_arxiv_id` do người gán nhãn quyết định sau khi đối chiếu title, authors, year, venue/arXiv ID.
- Mỗi publication cần ít nhất **20 matched pairs** và annotator cần ít nhất **100 matched pairs** tổng cộng.
- Ô `ref_arxiv_id` để trống nếu chưa làm/chưa chắc; ghi `none` chỉ khi đã kiểm tra và không có reference tương ứng.


In [ ]:
# Chỉ cần sửa CONFIG nếu đường dẫn/branch thay đổi.
CONFIG = {
    "REPO_URL": "https://github.com/VincesPowder/hcmus-project-arxiv.git",
    "BRANCH": "m3-manual-labeling",
    "REPO_DIR": "/content/hcmus-project-arxiv",
    "MSSV": "24127465",
    "M2_BIBTEX": "/content/drive/MyDrive/arXiv-metadata-hcmus/milestone2_outputs/m2_full_v5/exports/20261120-203957-9b2330/tables/bibtex",
    "REFERENCES": "/content/drive/MyDrive/arXiv-metadata-hcmus/2602.09870-2602.14869/work/tables/references.parquet",
    "SELECTION": "/content/drive/MyDrive/arXiv-metadata-hcmus/m3_selection/selection.csv",
    "OUT_DIR": "/content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465",
    "FILLED_DIR": "/content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled",
    "LABELS_OUT": "/content/hcmus-project-arxiv/labels/labels_manual_24127465.csv",
    "PAPERS": ["2602.11395", "2602.12886", "2602.11055", "2602.10677", "2602.11244"],
    "INSTALL_DEPENDENCIES": True,
    "USE_DRIVE": True,
    "MAKE_FRESH": False,
}

print("MSSV:", CONFIG["MSSV"])
print("Branch:", CONFIG["BRANCH"])
print("Papers:", CONFIG["PAPERS"])
print("Output:", CONFIG["OUT_DIR"])


## 1. Clone đúng branch và chuẩn bị môi trường

Cell này clone/fetch đúng branch `m3-manual-labeling`. Không dùng branch mặc định nếu branch M3 chưa được chọn.

Sau khi clone, notebook kiểm tra `src/label_sheet.py` tồn tại và kiểm tra syntax trước khi chạy.


In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

REPO = Path(CONFIG["REPO_DIR"])

def command(args, **kwargs):
    if not kwargs.get("capture_output"):
        with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
            status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, args)
        return None
    return subprocess.run(args, **kwargs)

if not REPO.exists():
    command(["git", "clone", "--branch", CONFIG["BRANCH"], CONFIG["REPO_URL"], str(REPO)])
else:
    print("Repo already exists:", REPO)
    current = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    print("Current branch:", current)
    if current != CONFIG["BRANCH"]:
        raise RuntimeError(
            f"Repo is on {current}, expected {CONFIG['BRANCH']}. "
            "Start with a fresh runtime or switch branch manually."
        )

os.chdir(REPO)

if CONFIG["INSTALL_DEPENDENCIES"]:
    req = REPO / "src" / "requirements-m2.txt"
    if req.is_file():
        command([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")

print("Python:", sys.version)
print("Repo:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Mount Google Drive và kiểm tra dữ liệu đầu vào

Các đường dẫn này đã được đặt theo output M2/M1 hiện tại của nhóm.

```text
M2 BibTeX:
milestone2_outputs/m2_full_v5/.../tables/bibtex

M1 references:
2602.09870-2602.14869/work/tables/references.parquet

M3 selection:
m3_selection/selection.csv
```

Nếu một đường dẫn không tồn tại, **dừng tại cell này** và sửa CONFIG trước khi chạy tiếp.


In [ ]:
if CONFIG["USE_DRIVE"]:
    from google.colab import drive
    print("Connecting Google Drive...", flush=True)
    drive.mount("/content/drive")
    print("Drive connected.", flush=True)

paths = {
    "M2_BIBTEX": Path(CONFIG["M2_BIBTEX"]),
    "REFERENCES": Path(CONFIG["REFERENCES"]),
    "SELECTION": Path(CONFIG["SELECTION"]),
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print("  exists:", path.exists(), "| dir:", path.is_dir(), "| file:", path.is_file())
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

print("All input paths exist.")


## 3. Kiểm tra `label_sheet.py` trước khi tạo CSV

Notebook yêu cầu phiên bản `label_sheet.py` có cột `best_suggestion`.

Schema mong đợi:

```text
arxiv_id
citation_key
entry_title
entry_authors
entry_year
entry_venue
best_suggestion
ref_arxiv_id
match_detail
note
```

Nếu thiếu `best_suggestion`, notebook sẽ dừng để tránh tạo CSV sai format.


In [ ]:
label_tool = REPO / "src" / "label_sheet.py"

if not label_tool.is_file():
    raise FileNotFoundError(f"Missing: {label_tool}")

command([sys.executable, "-m", "py_compile", str(label_tool)])
print("Syntax check: PASSED")

text = label_tool.read_text(encoding="utf-8")
if "best_suggestion" not in text:
    raise RuntimeError(
        "src/label_sheet.py does not contain best_suggestion. "
        "Update the file in the branch before running make."
    )

print("best_suggestion check: PASSED")
print("label_sheet.py:", label_tool)


## 4. Tạo các sheet gán nhãn (`make`)

Mỗi publication sẽ tạo:

- `<paper>.csv`: bảng BibTeX entry để người dùng gán nhãn.
- `<paper>_references.csv`: toàn bộ references của publication đó để đối chiếu.

Trong `<paper>.csv`, `best_suggestion` được ghi trực tiếp vào CSV. Nó chỉ là candidate/hint và không tự động điền `ref_arxiv_id`.

### Lưu ý trước khi chạy

Nếu đã có CSV **đã điền nhãn thủ công**, không bật `MAKE_FRESH` và không xóa output cũ. Chỉ tạo lại sheet khi chưa bắt đầu gán nhãn.


In [ ]:
OUT_DIR = Path(CONFIG["OUT_DIR"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

if CONFIG["MAKE_FRESH"]:
    if OUT_DIR.exists():
        shutil.rmtree(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print("Fresh output directory created:", OUT_DIR)

args = [
    sys.executable, "src/label_sheet.py", "make",
    "--mssv", CONFIG["MSSV"],
    "--bibtex", CONFIG["M2_BIBTEX"],
    "--references", CONFIG["REFERENCES"],
    "--selection", CONFIG["SELECTION"],
    "--out-dir", CONFIG["OUT_DIR"],
    "--papers", *CONFIG["PAPERS"],
]

command(args, cwd=REPO, env=env)
print("MAKE completed.")


## 5. Kiểm tra CSV sau khi `make`

Cell này kiểm tra:

1. đủ 5 publication;
2. có file `_references.csv` tương ứng;
3. CSV có cột `best_suggestion`;
4. `best_suggestion` không tự động điền vào `ref_arxiv_id`.


In [ ]:
import pandas as pd
from IPython.display import display

expected_columns = [
    "arxiv_id", "citation_key", "entry_title", "entry_authors", "entry_year",
    "entry_venue", "best_suggestion", "ref_arxiv_id", "match_detail", "note"
]

for paper in CONFIG["PAPERS"]:
    sheet = OUT_DIR / f"{paper}.csv"
    refs = OUT_DIR / f"{paper}_references.csv"

    print(f"
=== {paper} ===")
    print("sheet:", sheet.exists(), sheet)
    print("references:", refs.exists(), refs)

    if not sheet.exists():
        raise FileNotFoundError(sheet)
    if not refs.exists():
        raise FileNotFoundError(refs)

    df = pd.read_csv(sheet, dtype=str, keep_default_na=False)
    missing = [c for c in expected_columns if c not in df.columns]
    if missing:
        raise RuntimeError(f"{paper}: missing columns {missing}")

    print("entries:", len(df))
    print("columns:", df.columns.tolist())
    print("non-empty best_suggestion:", int((df["best_suggestion"].str.strip() != "").sum()))
    print("non-empty ref_arxiv_id:", int((df["ref_arxiv_id"].str.strip() != "").sum()))
    display(df.head(5))

print("CSV schema check: PASSED")


## 6. Hướng dẫn gán nhãn thủ công trên Google Sheets

Mở từng cặp file:

```text
2602.11395.csv
2602.11395_references.csv
```

### Với từng BibTeX entry

1. Đọc `entry_title`.
2. Xem `best_suggestion` để biết reference nào nên kiểm tra đầu tiên.
3. Tìm suggestion trong `<paper>_references.csv`.
4. Đối chiếu **title**.
5. Đối chiếu **authors**, ưu tiên first author/surname.
6. Đối chiếu **year**.
7. Kiểm tra thêm **venue, DOI hoặc arXiv ID** nếu có.
8. Nếu đúng, nhập **một** ID vào `ref_arxiv_id`.
9. Ghi lý do vào `match_detail`.
10. Nếu đã kiểm tra toàn bộ references mà không có tác phẩm tương ứng, nhập `none`.
11. Nếu chưa làm hoặc chưa chắc chắn, để `ref_arxiv_id` trống.

### Ví dụ

```text
best_suggestion: 2511.19116
ref_arxiv_id:    2511.19116
match_detail:    Title + first author + year matched
```

Nếu suggestion sai:

```text
best_suggestion: 2505.13632
ref_arxiv_id:    2504.10990
match_detail:    Title + authors + year matched
```

**Không sửa `best_suggestion` thành đáp án đúng.** Cột này giữ nguyên kết quả gợi ý của thuật toán.

### Google Sheets

Đặt các cột chứa arXiv ID (`arxiv_id`, `best_suggestion`, `ref_arxiv_id`) thành **Plain text** để tránh Sheets biến `2511.19116` thành dạng số như `251.119.116`.

Sau khi hoàn thành từng sheet: **File → Download → Comma-separated values (.csv)**.

Đặt các CSV đã điền vào:

```text
labeling/24127465_filled/
```

Không đưa các file `*_references.csv` vào thư mục `filled`.


## 7. Kiểm tra nhanh các file đã điền trước khi `collect`

Cell dưới đây chỉ đọc các CSV trong thư mục `filled`, không thay đổi chúng.

Mục tiêu là phát hiện sớm:

- thiếu publication;
- thiếu `best_suggestion`;
- chưa có dòng nào được gán nhãn;
- vẫn còn nhiều dòng `ref_arxiv_id` trống.


In [ ]:
FILLED_DIR = Path(CONFIG["FILLED_DIR"])
FILLED_DIR.mkdir(parents=True, exist_ok=True)

rows = []

for paper in CONFIG["PAPERS"]:
    f = FILLED_DIR / f"{paper}.csv"

    if not f.exists():
        rows.append({
            "arxiv_id": paper,
            "status": "MISSING",
            "entries": 0,
            "done": 0
        })
        continue

    df = pd.read_csv(f, dtype=str, keep_default_na=False)
    done = int((df["ref_arxiv_id"].str.strip() != "").sum()) if "ref_arxiv_id" in df.columns else 0

    rows.append({
        "arxiv_id": paper,
        "status": "OK",
        "entries": len(df),
        "done": done
    })

check = pd.DataFrame(rows)
display(check)
print("Total labelled rows:", int(check["done"].sum()))


## 8. Collect → kiểm tra → tạo `labels_manual_24127465.csv`

Chỉ chạy cell này sau khi đã hoàn thành các CSV trong `24127465_filled`.

`collect` sẽ kiểm tra `ref_arxiv_id` có thực sự thuộc references của **cùng publication** hay không, kiểm tra số matched pairs, và tạo labels cuối cùng.


In [ ]:
filled_files = sorted(
    p for p in FILLED_DIR.glob("*.csv")
    if not p.name.endswith("_references.csv")
)

print("Filled sheets:")
for p in filled_files:
    print(" -", p.name)

if not filled_files:
    raise RuntimeError("No filled CSV found. Complete Google Sheets first.")

collect_args = [
    sys.executable, "src/label_sheet.py", "collect",
    "--mssv", CONFIG["MSSV"],
    "--sheets-dir", CONFIG["FILLED_DIR"],
    "--references", CONFIG["REFERENCES"],
    "--selection", CONFIG["SELECTION"],
    "--out", CONFIG["LABELS_OUT"],
]

command(collect_args, cwd=REPO, env=env)
print("
COLLECT completed.")


## 9. Kiểm tra labels cuối cùng

Kết quả mong đợi:

- mỗi publication manual có ít nhất 20 matched pairs;
- tổng annotator có ít nhất 100 matched pairs;
- `none` được giữ như all-negative row;
- dòng chưa làm (blank) không được đưa vào labels cuối cùng.


In [ ]:
labels_path = Path(CONFIG["LABELS_OUT"])

if not labels_path.exists():
    raise FileNotFoundError(labels_path)

labels = pd.read_csv(labels_path, dtype=str, keep_default_na=False)

print("Labels file:", labels_path)
print("Rows:", len(labels))
print("Columns:", labels.columns.tolist())

if len(labels):
    summary = labels.assign(
        matched=labels["ref_arxiv_id"].str.strip() != ""
    ).groupby("arxiv_id").agg(
        labels=("citation_key", "count"),
        matched=("matched", "sum"),
        none=("ref_arxiv_id", lambda s: int((s.str.strip() == "").sum()))
    ).reset_index()

    display(summary)
    print("Total matched:", int(summary["matched"].sum()))
else:
    print("Labels file is empty.")


## 10. Checklist hoàn tất Milestone 3 manual labeling

- [ ] `src/label_sheet.py` có `best_suggestion`.
- [ ] `make` tạo đủ 5 `<paper>.csv`.
- [ ] Mỗi CSV có `best_suggestion`, `ref_arxiv_id`, `match_detail`, `note`.
- [ ] `best_suggestion` chỉ dùng làm gợi ý.
- [ ] `ref_arxiv_id` được người dùng xác nhận thủ công.
- [ ] `none` chỉ dùng sau khi đã kiểm tra references.
- [ ] Các CSV đã điền được tải vào `24127465_filled`.
- [ ] Không đưa `*_references.csv` vào `24127465_filled`.
- [ ] `collect` chạy thành công.
- [ ] Mỗi publication đạt >= 20 matched pairs.
- [ ] Tổng annotator đạt >= 100 matched pairs.
- [ ] Giữ lại các CSV đã điền để làm evidence (`match_detail`).

### Cấu trúc output cuối

```text
labeling/
├── 24127465/
│   ├── 2602.11395.csv
│   ├── 2602.11395_references.csv
│   ├── ...
│   └── 2602.11244_references.csv
│
└── 24127465_filled/
    ├── 2602.11395.csv
    ├── 2602.12886.csv
    ├── 2602.11055.csv
    ├── 2602.10677.csv
    └── 2602.11244.csv
```

Labels cuối cùng:

```text
/content/hcmus-project-arxiv/labels/labels_manual_24127465.csv
```
